# Data Storage Layer
## Big Data Analytics Project - Mobile Device Market Analysis

This notebook covers the **Data Storage Layer** of our Big Data Analytics project.

### Objectives:
- Store data in SQL database (PostgreSQL)
- Store data in NoSQL database (MongoDB)
- Communicate with databases (Import/Export)
- Merge multiple data sources appropriately
- Exploit specific properties of each database

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from datetime import datetime
import json

# Database connections
from src.db_connect import get_connection
from src.mongodb_connect import get_mongodb_connection

print("SUCCESS: All imports successful!")

SUCCESS: All imports successful!


## 2. Load Raw Data

In [2]:
# Load data from Data Source Layer
raw_df = pd.read_csv("../data/raw_merged_data.csv")

print(f"Loaded {len(raw_df)} records")
print(f"Columns: {list(raw_df.columns)}")
raw_df.head()

Loaded 1359 records
Columns: ['Unnamed: 0', 'device_name', 'brand', 'Model', 'Battery capacity (mAh)', 'screen_size', 'Touchscreen', 'Resolution x', 'Resolution y', 'Processor', 'storage', 'Rear camera', 'Front camera', 'os', 'Wi-Fi', 'Bluetooth', 'GPS', 'Number of SIMs', '3G', '4G/ LTE', 'price', 'ram']


,Unnamed: 0,device_name,brand,Model,Battery capacity (mAh),screen_size,Touchscreen,Resolution x,Resolution y,Processor,...,Front camera,os,Wi-Fi,Bluetooth,GPS,Number of SIMs,3G,4G/ LTE,price,ram
0,0,OnePlus 7T Pro McLaren Edition,OnePlus,7T Pro McLaren Edition,4085,6.67,Yes,1440,3120,8,...,16.0,Android,Yes,Yes,Yes,2,Yes,Yes,58998,11.718750
1,1,Realme X2 Pro,Realme,X2 Pro,4000,6.50,Yes,1080,2400,8,...,16.0,Android,Yes,Yes,Yes,2,Yes,Yes,27999,5.859375
2,2,iPhone 11 Pro Max,Apple,iPhone 11 Pro Max,3969,6.50,Yes,1242,2688,6,...,12.0,iOS,Yes,Yes,Yes,2,Yes,Yes,106900,3.906250
3,3,iPhone 11,Apple,iPhone 11,3110,6.10,Yes,828,1792,6,...,12.0,iOS,Yes,Yes,Yes,2,Yes,Yes,62900,3.906250
4,4,LG G8X ThinQ,LG,G8X ThinQ,4000,6.40,Yes,1080,2340,8,...,32.0,Android,Yes,Yes,Yes,1,No,No,49990,5.859375


## 3. PostgreSQL (SQL Database)

Store structured data in PostgreSQL relational database.

In [3]:
# Connect to PostgreSQL
pg_conn = get_connection()

if pg_conn:
    print("SUCCESS: Connected to PostgreSQL")
else:
    print("ERROR: Failed to connect to PostgreSQL")

SUCCESS: Connection successful!
Database: neondb
User: neondb_owner
Server time: 2026-01-08 16:41:18.590062+00:00 

SUCCESS: Connected to PostgreSQL


### Create Tables and Schema

In [4]:
if pg_conn:
    cur = pg_conn.cursor()
    
    # Create devices table
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS mobile_devices (
        id SERIAL PRIMARY KEY,
        device_name VARCHAR(255),
        brand VARCHAR(100),
        price DECIMAL(10, 2),
        screen_size DECIMAL(5, 2),
        ram INTEGER,
        storage INTEGER,
        os VARCHAR(50),
        release_date VARCHAR(20),
        category VARCHAR(50),
        source VARCHAR(50),
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """
    
    cur.execute(create_table_sql)
    pg_conn.commit()
    print("SUCCESS: Table 'mobile_devices' created")
    
    # Clear existing data (optional)
    cur.execute("TRUNCATE TABLE mobile_devices CASCADE;")
    pg_conn.commit()
    
    cur.close()

SUCCESS: Table 'mobile_devices' created


### Import Data to PostgreSQL

In [5]:
if pg_conn:
    # Prepare data for insertion
    insert_sql = """
    INSERT INTO mobile_devices 
    (device_name, brand, price, screen_size, ram, storage, os, release_date, category, source)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    
    cur = pg_conn.cursor()
    
    # Insert records
    records_inserted = 0
    for _, row in raw_df.iterrows():
        try:
            values = (
                row.get('device_name', None),
                row.get('brand', None),
                row.get('price', None),
                row.get('screen_size', None),
                row.get('ram', None),
                row.get('storage', None),
                row.get('os', None),
                row.get('release_date', None),
                row.get('category', None),
                row.get('source', 'unknown')
            )
            cur.execute(insert_sql, values)
            records_inserted += 1
        except Exception as e:
            print(f"WARNING: Error inserting record: {e}")
            continue
    
    pg_conn.commit()
    cur.close()
    
    print(f"SUCCESS: Inserted {records_inserted} records into PostgreSQL")

SUCCESS: Inserted 1359 records into PostgreSQL


### Query Data from PostgreSQL

In [6]:
if pg_conn:
    # Export data using SQL query
    query = """
    SELECT device_name, brand, price, screen_size, ram, storage, os, category
    FROM mobile_devices
    ORDER BY price DESC
    LIMIT 10;
    """
    
    pg_df = pd.read_sql_query(query, pg_conn)
    print("Top 10 devices by price (from PostgreSQL):")
    print(pg_df)

Top 10 devices by price (from PostgreSQL):
                device_name    brand     price  screen_size  ram  storage  \
0     Samsung Galaxy Z Flip  Samsung  174990.0         6.70    8      256   
1       Samsung Galaxy Fold  Samsung  164999.0         7.30   12      512   
2         iPhone 11 Pro Max    Apple  106900.0         6.50    4       64   
3             iPhone 11 Pro    Apple   96900.0         5.80    4       64   
4  Samsung Galaxy S20 Ultra  Samsung   92999.0         6.90   12      128   
5               HP Elite x3       HP   88719.0         5.96    4       64   
6         Google Pixel 4 XL   Google   83900.0         6.30    6       64   
7   Samsung Galaxy Note 10+  Samsung   79699.0         6.80   12      256   
8        Huawei Mate 30 Pro   Huawei   77299.0         6.53    8      128   
9                  HTC U12+      HTC   74990.0         6.00    6       64   

        os category  
0  Android     None  
1  Android     None  
2      iOS     None  
3      iOS     None  

C:\Users\User\AppData\Local\Temp\ipykernel_15920\136450101.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pg_df = pd.read_sql_query(query, pg_conn)


## 4. MongoDB (NoSQL Database)

Store semi-structured and flexible data in MongoDB NoSQL database.

In [7]:
# Connect to MongoDB
mongodb_db = get_mongodb_connection()

if mongodb_db is not None:
    print("SUCCESS: Connected to MongoDB")
else:
    print("WARNING: MongoDB not available - using local fallback")
    print("Note: Install MongoDB or set MONGODB_URI in .env file")

SUCCESS: MongoDB connection successful!
Database: bigdata_analytics

SUCCESS: Connected to MongoDB


### Import Data to MongoDB

In [8]:
if mongodb_db is not None:
    # Get collection
    devices_collection = mongodb_db['mobile_devices']
    
    # Clear existing data
    devices_collection.delete_many({})
    
    # Convert DataFrame to list of dictionaries
    documents = raw_df.to_dict('records')
    
    # Add metadata
    for doc in documents:
        doc['stored_at'] = datetime.now().isoformat()
        doc['database'] = 'mongodb'
    
    # Insert documents
    result = devices_collection.insert_many(documents)
    
    print(f"SUCCESS: Inserted {len(result.inserted_ids)} documents into MongoDB")
    print(f"Collection: {devices_collection.name}")

SUCCESS: Inserted 1359 documents into MongoDB
Collection: mobile_devices


### Query Data from MongoDB (NoSQL Advantages)

In [9]:
if mongodb_db is not None:
    devices_collection = mongodb_db['mobile_devices']
    
    # Example 1: Find premium devices
    premium_devices = list(devices_collection.find({"category": "Premium"}))
    print(f"Found {len(premium_devices)} premium devices")
    
    # Example 2: Aggregate query (MongoDB aggregation pipeline)
    pipeline = [
        {"$group": {
            "_id": "$brand",
            "avg_price": {"$avg": "$price"},
            "count": {"$sum": 1}
        }},
        {"$sort": {"avg_price": -1}}
    ]
    
    brand_stats = list(devices_collection.aggregate(pipeline))
    print("\nBrand Statistics (MongoDB Aggregation):")
    for stat in brand_stats:
        print(f"  {stat['_id']}: {stat['count']} devices, Avg Price: ${stat['avg_price']:.2f}")

Found 0 premium devices

Brand Statistics (MongoDB Aggregation):
  HP: 1 devices, Avg Price: $88719.00
  Razer: 1 devices, Avg Price: $54990.00
  Apple: 17 devices, Avg Price: $45510.47
  Cat: 4 devices, Avg Price: $45219.75
  Huawei: 12 devices, Avg Price: $38564.75
  Google: 10 devices, Avg Price: $38166.40
  OnePlus: 12 devices, Avg Price: $32908.75
  Black Shark: 1 devices, Avg Price: $29999.00
  Zuk: 1 devices, Avg Price: $29124.00
  Sony: 28 devices, Avg Price: $28315.54
  Samsung: 101 devices, Avg Price: $22500.07
  HTC: 35 devices, Avg Price: $19297.14
  Microsoft: 5 devices, Avg Price: $17755.40
  Nubia: 12 devices, Avg Price: $17754.25
  Oppo: 35 devices, Avg Price: $15511.26
  Poco: 2 devices, Avg Price: $15494.50
  Asus: 37 devices, Avg Price: $14926.46
  LG: 42 devices, Avg Price: $14884.45
  Meizu: 9 devices, Avg Price: $14710.89
  BlackBerry: 17 devices, Avg Price: $14180.88
  Vivo: 52 devices, Avg Price: $12501.92
  Honor: 33 devices, Avg Price: $12056.67
  Realme: 18 d

## 5. Save Processed Data for Next Layer

In [10]:
# Save cleaned data for Processing Layer
if pg_conn:
    # Export from PostgreSQL
    final_query = "SELECT * FROM mobile_devices;"
    processed_df = pd.read_sql_query(final_query, pg_conn)
    
    # Save to CSV
    output_path = "../data/stored_data.csv"
    processed_df.to_csv(output_path, index=False)
    print(f"SUCCESS: Processed data saved to {output_path}")
    
    # Close connection
    pg_conn.close()
    print("PostgreSQL connection closed")


C:\Users\User\AppData\Local\Temp\ipykernel_15920\4082980112.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  processed_df = pd.read_sql_query(final_query, pg_conn)


SUCCESS: Processed data saved to ../data/stored_data.csv
PostgreSQL connection closed


## Summary

**Data Storage Layer Complete**

- Stored data in PostgreSQL (SQL)
- Stored data in MongoDB (NoSQL)
- Imported/Exported data from both databases
- Exploited specific properties of each database
- Merged data sources appropriately

